In [1]:
import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [2]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
# from siuba import *

fs = get_fs()

In [3]:
import altair as alt
import folium
import matplotlib.pyplot as plt


In [4]:
from IPython.display import HTML

In [5]:
pd.set_option("display.max_columns", None)

In [6]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"

In [63]:
shape_data_name = "replica-stm_scag_origins-origin_layer.zip"

origins_name = "replica-mtc_scag_origins-origin_layer.zip"
shape_destinations_name = "replica-mtc_scag_destinations-destination_layer.zip"

#### Check Shape Data -- Hex Cells

* When you create a study in Replica, you can download the geography breakdown that is available within the map view
* We want to change that geography breakdown to `H3 Hex Cells Resolution 7 (~2 sq mi/cell)`
* Set up your analysis as stated in the Analysis Steps
* Download the Hex Cells once you have the top 50 trip hex cells
* Upload to GCS and follow the steps below before uploading to Replica as a Custom Geographies

In [73]:
# with get_fs().open(f"{gcs_path}{shape_data_name}") as f:
#         shps = to_snakecase(gpd.read_file(f))

# with get_fs().open(f"{gcs_path}origins/{shape_destinations_name}") as f:
#         shps = to_snakecase(gpd.read_file(f))

with get_fs().open(f"{gcs_path}origins/{origins_name}") as f:
        shps = to_snakecase(gpd.read_file(f))

In [74]:
assert (len(shps) == 50)

In [75]:
shps.sample()

,geoname,customgeo,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
22,872830812ffffff,608692970350182400,37.82,-122.2488,2.072,7695,3713.7247,"POLYGON ((-122.23995 37.80942, -122.25629 37.8..."


In [76]:
shps = shps.set_geometry("geometry")

In [77]:
# (shps.sample()).explore()

* Rename the columns to match Replica's Custom Geographies specs

In [78]:
shps = shps.rename(columns={"geoname":"name", "customgeo":"id"})

In [79]:
shps.sample()

,name,id,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
39,87283098bffffff,608692976675192800,37.7883,-122.2223,2.0733,5437,2622.339,"POLYGON ((-122.21352 37.77777, -122.22986 37.7..."


In [80]:
geojson_string = shps.to_json()

* Run the cell below `geojson_string`
* Copy and paste the string in a new file
* Rename the file "Shps_{County}_Origins.geojson"
* Download the new geojson and upload to Replica Custom Geographies
* Use the newly uploaded Geography as the custom geography for the Trip Data Download (steps detailed in Analysis Steps)

In [81]:
geojson_string

'{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {"name": "87283082affffff", "id": 608692970752835600, "centrlat": 37.7865, "centrlon": -122.3945, "sqmi": 2.0699, "origin": 54041, "orig_sqmi": 26107.4026}, "geometry": {"type": "Polygon", "coordinates": [[[-122.385634554, 37.776004201], [-122.401953852, 37.774760714], [-122.410770923, 37.785293474], [-122.403268745, 37.797070861], [-122.386946098, 37.798315483], [-122.378128979, 37.787781581], [-122.385634554, 37.776004201]]]}}, {"id": "1", "type": "Feature", "properties": {"name": "872830828ffffff", "id": 608692970719281200, "centrlat": 37.7735, "centrlon": -122.4183, "sqmi": 2.0698, "origin": 30692, "orig_sqmi": 14828.5789}, "geometry": {"type": "Polygon", "coordinates": [[[-122.409454632, 37.762981848], [-122.425769087, 37.761735734], [-122.434586108, 37.77226735], [-122.427088725, 37.784046223], [-122.410770923, 37.785293474], [-122.401953852, 37.774760714], [-122.409454632, 37.762981848]]]}},

In [50]:
saved = to_snakecase(gpd.read_file("./saved-selection-geo-stm-scag-top50-origins_h3z7.geojson"))


In [52]:
# saved

##### Checking to see if geonames match

In [21]:
shps[shps['id']==608718595349807100]

,name,id,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
0,8729a565bffffff,608718595349807100,33.9337,-118.4119,2.2254,52256,23481.8963,"POLYGON ((-118.40364 33.92238, -118.41978 33.9..."


#### Check Trips Data

In [17]:
df = to_snakecase(pd.read_csv(f"{gcs_path}{origins_name}"))  

In [18]:
df.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,tour_type,trip_start_local_hour,trip_end_local_hour,origin_custom_id,destination_custom_id,origin_zcta_2020,destination_zcta_2020,destination_custom_lat,destination_custom_lng,origin_custom_lng,origin_custom_lat
236186,18333842258138891537,"2 (Tract 2653.01, Los Angeles, CA)","2653.01 (Los Angeles, CA)","Los Angeles County, CA",California,"4 (Tract 320.02, Orange, CA)","320.02 (Orange, CA)","Orange County, CA",California,private_auto,home,work,14:28:00,16:11:12,103,54.0,NaN,NaN,NaN,NaN,NaN,education,education,single_family,single_family,1894194950998001667,17575166101483736090,51.0,female,white_not_hispanic_or_latino,employed,in_person,81142.0,private_auto,3,255020.0,three_plus,core,naics61,single_family,not_attending_school,bachelors_degree,owner,english,"4 (Tract 320.02, Orange, CA)","320.02 (Orange, CA)","Orange County, CA",California,"2 (Tract 2653.01, Los Angeles, CA)","2653.01 (Los Angeles, CA)","Los Angeles County, CA",California,8729a199dffffff,NaN,commute,14,16,608718334464098300,NaN,90095,92691,NaN,NaN,-118.4387,34.0603


In [19]:
df.origin_custom_id.value_counts()

origin_custom_id
608718595349807100    52256
608718350972878800    38886
608718350654111700    29935
608718334464098300    26434
608718350670889000    23046
608718350788329500    19634
608718334682202100    17933
608718350838661100    16634
608718334212440000    16107
608718334480875500    15325
608718333994336300    15238
608718588957687800    15061
608718334447321100    15019
608718334849974300    14666
608718350721220600    14614
608718350167572500    14582
608718350905770000    14525
608718594158624800    14431
608718350687666200    14302
608718272824606700    13977
608718334413766700    13532
608718345268625400    13164
608718350637334500    12771
608718595333029900    12770
608718329984581600    12712
608718350888992800    12047
608718334782865400    11822
608718334027890700    11485
608718350620557300    10713
608718334178885600    10570
608718350754775000    10452
608718597463736300    10025
608718334011113500     9690
608718237978329100     9663
608718595400138800     9538
608